# Stage 4 — Locked Internal Robustness and Publication Readiness

**Scientific status:** post hoc robustness analysis. Stage 3B common-core ridge remains the frozen confirmatory primary analysis. Stage 3C remains exploratory. Stage 4 cannot promote or replace either analysis.

This notebook implements the next defensible step when external data are not yet available:

1. A **fully nested family-selector audit** using saved Stage 3C inner scores and untouched outer-test predictions.
2. Locked **joint/anatomy-domain removal** analyses for `NoHandJointInvolvement` and `AsymmetricArthritis`, limited to broad ridge and XGBoost.
3. Missingness and subgroup heterogeneity audits without subgroup refitting.
4. Publication-grade calibration, stability, selector, and domain-removal figures.
5. Fine-grained signed checkpoints, technical audits, and a final self-contained HTML report.

No new algorithm families, tuning-grid expansion, threshold optimization, resampling, class weighting, or favorable seed search are permitted.

In [ ]:
# 1. Imports, versions, and deterministic settings
from __future__ import annotations
import hashlib, json, math, os, pickle, platform, subprocess, sys, time, warnings
from dataclasses import dataclass
from pathlib import Path
from packaging.version import Version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import yaml
from IPython.display import display, Markdown
from scipy.special import expit, logit

import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier

SKLEARN_VERSION=Version(sklearn.__version__)
if SKLEARN_VERSION < Version('1.8'):
    raise RuntimeError(f'scikit-learn >=1.8,<1.10 required; found {sklearn.__version__}')

MASTER_SEED=20261020
np.random.seed(MASTER_SEED)
pd.set_option('display.max_columns',180); pd.set_option('display.max_rows',180)
pd.set_option('display.width',220); pd.set_option('display.float_format',lambda x:f'{x:,.5f}')

mpl.rcParams.update({'figure.dpi':130,'savefig.dpi':400,'font.size':10,'axes.titlesize':12,
                     'axes.labelsize':10,'legend.fontsize':9,'figure.titlesize':14,
                     'axes.spines.top':False,'axes.spines.right':False,'svg.fonttype':'none'})
sns.set_theme(style='whitegrid',context='paper',palette='colorblind')
print('Python:',sys.version.split()[0]); print('Platform:',platform.platform())
print('pandas:',pd.__version__,'| scikit-learn:',sklearn.__version__)

In [ ]:
# 2. Project discovery, execution mode, and isolated outputs
NOTEBOOK_NAME='04_locked_internal_robustness.ipynb'
ENV_PROJECT = Path(os.environ['RA_PROJECT_DIR']).expanduser() if os.environ.get('RA_PROJECT_DIR') else Path.cwd()
COLAB_PROJECT=Path.cwd()
env_root=os.environ.get('RA_PROJECT_DIR')
candidates=[Path(env_root)] if env_root else []
candidates += [ENV_PROJECT,COLAB_PROJECT,Path.cwd(),Path.cwd()/'ra_analysis',
               Path.cwd()/'stage2_outputs',Path.cwd()]
PROJECT_DIR=next((p for p in candidates if (p/'derived_data').exists() and (p/'modeling_results').exists()),None)
if PROJECT_DIR is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_DIR=next((p for p in candidates if (p/'derived_data').exists() and (p/'modeling_results').exists()),None)
    except (ImportError,ModuleNotFoundError): pass
if PROJECT_DIR is None: raise FileNotFoundError('Set RA_PROJECT_DIR to the directory containing derived_data and modeling_results.')

DERIVED_DIR=PROJECT_DIR/'derived_data'
STAGE3B_DIR=PROJECT_DIR/'modeling_results'; STAGE3B_TABLES=STAGE3B_DIR/'tables'
STAGE3C_DIR=PROJECT_DIR/'modeling_results_stage3c'
STAGE3C_TABLES=next((p for p in [STAGE3C_DIR/'tables',STAGE3C_DIR/'tables-3c'] if p.exists()),None)
if STAGE3C_TABLES is None: raise FileNotFoundError('Stage 3C tables or tables-3c directory not found')

protocol_candidates=[PROJECT_DIR/'stage4_robustness_protocol.yaml',PROJECT_DIR/'config'/'stage4_robustness_protocol.yaml',
                     PROJECT_DIR.parent/'stage4_robustness_protocol.yaml',Path.cwd()/'stage4_robustness_protocol.yaml',
                     ENV_PROJECT/'stage4_robustness_protocol.yaml']
PROTOCOL_PATH=next((p for p in protocol_candidates if p.exists()),None)
if PROTOCOL_PATH is None: raise FileNotFoundError('stage4_robustness_protocol.yaml not found')
with open(PROTOCOL_PATH,encoding='utf-8') as f: PROTOCOL=yaml.safe_load(f)

OUT_DIR=PROJECT_DIR/'modeling_results_stage4'
TABLE_DIR=OUT_DIR/'tables'; FIGURE_DIR=OUT_DIR/'figures'; CHECKPOINT_DIR=OUT_DIR/'checkpoints'; REPORT_DIR=OUT_DIR/'report'
for d in [OUT_DIR,TABLE_DIR,FIGURE_DIR,CHECKPOINT_DIR,REPORT_DIR]: d.mkdir(parents=True,exist_ok=True)

QUICK_MODE=os.environ.get('RA_STAGE4_QUICK_TEST','0')=='1'
FORCE_RECOMPUTE=os.environ.get('RA_STAGE4_FORCE_RECOMPUTE','0')=='1'
XGB_DEVICE=os.environ.get('RA_STAGE4_XGB_DEVICE','cpu')
ACTIVE_REPEATS=[0] if QUICK_MODE else list(range(10))
OUTER_FOLDS=3 if QUICK_MODE else 5
INNER_SPLITS=2 if QUICK_MODE else 5
BOOTSTRAPS=100 if QUICK_MODE else 2000
CALIBRATION_BOOTSTRAPS=30 if QUICK_MODE else 500
SUBGROUP_BOOTSTRAPS=30 if QUICK_MODE else 300
XGB_SEARCH_ITER=4 if QUICK_MODE else 50
N_JOBS=-1
OUTCOMES=['NegRFandACCP','NoHandJointInvolvement','AsymmetricArthritis']
CANDIDATES=['refined_broad_ridge','broad_lasso','refined_broad_elastic_net','linear_svc','rbf_svc','random_forest','xgboost','adaboost']
print('Project:',PROJECT_DIR.resolve()); print('Stage 3C tables:',STAGE3C_TABLES.resolve())
print('Stage 4 output:',OUT_DIR.resolve()); print('Mode:','QUICK SOFTWARE TEST' if QUICK_MODE else 'FULL LOCKED ROBUSTNESS')

## Input integrity and signed run definition

In [ ]:
# 3. Load immutable Stage 3B/3C artifacts and define a signed Stage 4 run
def sha256_file(path:Path)->str:
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

required={
 'splits':STAGE3B_TABLES/'saved_outer_split_assignments.csv',
 'stage3b_predictions':STAGE3B_TABLES/'patient_averaged_oof_predictions.csv',
 'stage3b_audit':STAGE3B_TABLES/'stage3b_validation_audit.csv',
 'stage3c_repeated':STAGE3C_TABLES/'all_repeated_oof_predictions.csv',
 'stage3c_averaged':STAGE3C_TABLES/'patient_averaged_oof_predictions.csv',
 'stage3c_hyperparameters':STAGE3C_TABLES/'selected_hyperparameters.csv',
 'stage3c_audit':STAGE3C_TABLES/'stage3c_validation_audit.csv'}
for label,p in required.items():
    if not p.exists(): raise FileNotFoundError(f'{label}: {p}')
for p in [required['stage3b_audit'],required['stage3c_audit']]:
    a=pd.read_csv(p)
    if a.empty or not a.status.eq('PASS').all(): raise AssertionError(f'Prior audit failed: {p}')

saved_splits=pd.read_csv(required['splits'])
stage3b_avg=pd.read_csv(required['stage3b_predictions'])
stage3c_repeated=pd.read_csv(required['stage3c_repeated'])
stage3c_avg=pd.read_csv(required['stage3c_averaged'])
stage3c_hp=pd.read_csv(required['stage3c_hyperparameters'])
for frame in [saved_splits,stage3c_repeated,stage3c_hp]:
    if 'repeat' in frame: frame.query('repeat in @ACTIVE_REPEATS',inplace=True)

signature_payload={'pipeline_version':'4.1-locked-robustness','mode':'quick' if QUICK_MODE else 'full',
                   'protocol_sha256':sha256_file(PROTOCOL_PATH),'inputs':{k:sha256_file(v) for k,v in required.items()},
                   'active_repeats':ACTIVE_REPEATS,'outer_folds':OUTER_FOLDS,'inner_splits':INNER_SPLITS,'bootstraps':BOOTSTRAPS,
                   'xgb_search_iter':XGB_SEARCH_ITER,'seed':MASTER_SEED}
RUN_SIGNATURE=hashlib.sha256(json.dumps(signature_payload,sort_keys=True).encode()).hexdigest()
with open(OUT_DIR/'run_signature.json','w',encoding='utf-8') as f:
    json.dump({'run_signature':RUN_SIGNATURE,**signature_payload},f,indent=2)
print('Run signature:',RUN_SIGNATURE)
print('Active repeated predictions:',stage3c_repeated.shape,'| hyperparameter rows:',stage3c_hp.shape)

## A. Fully nested model-family selector

For each outer split, the family is selected exclusively by the saved inner-CV log loss from that outer training set. Its already-generated untouched outer-test predictions are then used. This estimates the performance of the **selection procedure**, not the optimistic maximum observed after external comparison of families.

In [ ]:
# 4. Construct untouched outer-test predictions for the nested selector
hp_candidates=stage3c_hp[stage3c_hp.model.isin(CANDIDATES)].copy()
expected=3*len(ACTIVE_REPEATS)*OUTER_FOLDS*len(CANDIDATES)
if len(hp_candidates)!=expected: raise AssertionError((len(hp_candidates),expected))
selector_choices=(hp_candidates.loc[hp_candidates.groupby(['outcome','repeat','fold']).best_inner_log_loss.idxmin(),
                                    ['outcome','repeat','fold','model','best_inner_log_loss','best_params_json']]
                  .rename(columns={'model':'selected_family'}))
selector_repeated=(stage3c_repeated.merge(selector_choices,on=['outcome','repeat','fold'],validate='many_to_one')
                   .query('model==selected_family').copy())
selector_repeated['model']='nested_family_selector'
selector_repeated=selector_repeated[['VisitID','row_index','outcome','model','repeat','fold','y','probability']]
expected_rows=3*len(ACTIVE_REPEATS)*820
if len(selector_repeated)!=expected_rows: raise AssertionError((len(selector_repeated),expected_rows))
selector_frequency=(selector_choices.groupby(['outcome','selected_family']).size().rename('outer_splits').reset_index())
selector_frequency['selection_fraction']=selector_frequency.outer_splits/(OUTER_FOLDS*len(ACTIVE_REPEATS))
selector_choices.to_csv(TABLE_DIR/'nested_selector_choices_by_outer_split.csv',index=False,encoding='utf-8-sig')
selector_frequency.to_csv(TABLE_DIR/'nested_selector_family_frequency.csv',index=False,encoding='utf-8-sig')
display(selector_frequency.sort_values(['outcome','outer_splits'],ascending=[True,False]))

In [ ]:
# 5. Metrics and patient-level uncertainty helpers
EPS=1e-8
def calibration_stats(y,p):
    p=np.clip(np.asarray(p,float),EPS,1-EPS); y=np.asarray(y,int)
    if np.std(logit(p))<1e-10: return np.nan,np.nan
    try:
        m=LogisticRegression(C=np.inf,l1_ratio=0.0,solver='lbfgs',max_iter=5000).fit(logit(p).reshape(-1,1),y)
        return float(m.intercept_[0]),float(m.coef_[0,0])
    except Exception: return np.nan,np.nan

def metric_dict(y,p):
    ci,cs=calibration_stats(y,p)
    return {'auroc':roc_auc_score(y,p),'average_precision':average_precision_score(y,p),
            'log_loss':log_loss(y,p,labels=[0,1]),'brier':brier_score_loss(y,p),
            'calibration_intercept':ci,'calibration_slope':cs}

def average_repeats(frame):
    return (frame.groupby(['outcome','model','VisitID','y'],as_index=False)
            .agg(probability=('probability','mean'),oof_repetitions=('probability','size'),
                 probability_sd_across_repeats=('probability','std')))

def bootstrap_metrics(frame,n_boot,seed):
    y=frame.y.to_numpy(int);p=frame.probability.to_numpy(float);rng=np.random.default_rng(seed)
    pos=np.flatnonzero(y==1);neg=np.flatnonzero(y==0); rows=[]
    for _ in range(n_boot):
        idx=np.r_[rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)]
        rows.append(metric_dict(y[idx],p[idx]))
    return pd.DataFrame(rows)

def paired_difference(a,b,n_boot,seed):
    m=a.merge(b,on=['outcome','VisitID','y'],suffixes=('_a','_b'),validate='one_to_one')
    y=m.y.to_numpy(int);pa=m.probability_a.to_numpy();pb=m.probability_b.to_numpy()
    pos=np.flatnonzero(y==1);neg=np.flatnonzero(y==0);rng=np.random.default_rng(seed)
    def one(idx):
        ma=metric_dict(y[idx],pa[idx]);mb=metric_dict(y[idx],pb[idx])
        return {'auroc':ma['auroc']-mb['auroc'],'average_precision':ma['average_precision']-mb['average_precision'],
                'log_loss':mb['log_loss']-ma['log_loss'],'brier':mb['brier']-ma['brier']}
    est=one(np.arange(len(y))); vals={k:[] for k in est}
    for _ in range(n_boot):
        idx=np.r_[rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)]
        z=one(idx)
        for k,v in z.items(): vals[k].append(v)
    return pd.DataFrame([{'metric':k,'improvement_over_reference':v,'ci_low':np.quantile(vals[k],.025),
                          'ci_high':np.quantile(vals[k],.975)} for k,v in est.items()])

In [ ]:
# 6. Selector estimates, intervals, and paired comparisons
selector_avg=average_repeats(selector_repeated)
original_active=average_repeats(stage3c_repeated)
selector_perf=[]; selector_ci=[]; selector_pairs=[]
for outcome,g in selector_avg.groupby('outcome'):
    m=metric_dict(g.y,g.probability); selector_perf.append({'outcome':outcome,'model':'nested_family_selector','n':len(g),'events':int(g.y.sum()),**m})
    boot=bootstrap_metrics(g,BOOTSTRAPS,MASTER_SEED+OUTCOMES.index(outcome))
    for metric,value in m.items():
        v=boot[metric].dropna(); selector_ci.append({'outcome':outcome,'model':'nested_family_selector','metric':metric,
            'estimate':value,'ci_low':v.quantile(.025),'ci_high':v.quantile(.975),'bootstrap_resamples':BOOTSTRAPS})
    a=g[['outcome','VisitID','y','probability']]
    b=original_active.query("outcome==@outcome and model=='refined_broad_ridge'")[['outcome','VisitID','y','probability']]
    d=paired_difference(a,b,BOOTSTRAPS,MASTER_SEED+100+OUTCOMES.index(outcome));
    d.insert(0,'reference','refined_broad_ridge');d.insert(0,'comparator','nested_family_selector');d.insert(0,'outcome',outcome)
    selector_pairs.append(d)
selector_perf=pd.DataFrame(selector_perf);selector_ci=pd.DataFrame(selector_ci);selector_pairs=pd.concat(selector_pairs,ignore_index=True)
selector_avg.to_csv(TABLE_DIR/'nested_selector_patient_averaged_predictions.csv',index=False,encoding='utf-8-sig')
selector_perf.to_csv(TABLE_DIR/'nested_selector_performance.csv',index=False,encoding='utf-8-sig')
selector_ci.to_csv(TABLE_DIR/'nested_selector_bootstrap_ci.csv',index=False,encoding='utf-8-sig')
selector_pairs.to_csv(TABLE_DIR/'nested_selector_paired_vs_refined_ridge.csv',index=False,encoding='utf-8-sig')
display(selector_perf);display(selector_pairs)

In [ ]:
# 7. Publication figure: selector family frequency and paired performance
family_order=CANDIDATES
colors=dict(zip(family_order,sns.color_palette('colorblind',len(family_order))))
fig,axes=plt.subplots(1,2,figsize=(14,5.2))
pivot=(selector_frequency.pivot(index='outcome',columns='selected_family',values='selection_fraction')
       .reindex(OUTCOMES).reindex(columns=family_order).fillna(0))
left=np.zeros(len(pivot))
for family in family_order:
    axes[0].barh(pivot.index,pivot[family],left=left,label=family,color=colors[family],edgecolor='white',linewidth=.5)
    left+=pivot[family].to_numpy()
axes[0].set(xlabel='Fraction of outer splits selected',ylabel='',xlim=(0,1),title='A. Family selected by inner-CV log loss')
axes[0].legend(loc='upper center',bbox_to_anchor=(.5,-.18),ncol=2,frameon=False)
plot=selector_pairs[selector_pairs.metric.isin(['auroc','log_loss'])].copy();plot['label']=plot.outcome+' | '+plot.metric
for i,r in plot.reset_index(drop=True).iterrows():
    axes[1].plot([r.ci_low,r.ci_high],[i,i],color='#2c7fb8',lw=2);axes[1].scatter(r.improvement_over_reference,i,color='#d95f0e',s=32,zorder=3)
axes[1].axvline(0,color='black',ls='--',lw=1);axes[1].set_yticks(range(len(plot)),plot.label)
axes[1].set(xlabel='Improvement over refined broad ridge\n(positive favors selector)',ylabel='',title='B. Paired patient-bootstrap differences')
fig.suptitle('Fully nested model-family selection audit');fig.tight_layout()
for ext in ['png','svg']: fig.savefig(FIGURE_DIR/f'nested_family_selector_audit.{ext}',bbox_inches='tight',dpi=400 if ext=='png' else None)
plt.show();plt.close(fig)

## B. Locked joint/anatomy-domain removal

Only the two prespecified models and two outcomes are refitted. Every outer fold receives its own signed checkpoint, so an interrupted run resumes without repeating completed folds. The same Stage 3B outer assignments and fold-contained preprocessing are retained.

In [ ]:
# 8. Frozen ablation definition and leakage-controlled preprocessing
ABLATION_OUTCOMES=['NoHandJointInvolvement','AsymmetricArthritis']
ABLATION_MODELS=['refined_broad_ridge','xgboost']
REMOVE_EXACT=set(PROTOCOL['semantic_ablations']['remove_exact'])
REMOVE_CONTAINS=tuple(PROTOCOL['semantic_ablations']['remove_name_contains'])
CATEGORY_LEVELS={'Gender':['Female','Male'],'RAOnset':['Insidious','Acute','Unknown'],'PhysicalActivity':['Moderate','Low','High']}

class MissingnessFlags(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None): self.n_features_in_=X.shape[1]; return self
    def transform(self,X): return pd.isna(X).astype(float).to_numpy() if hasattr(X,'to_numpy') else np.isnan(X).astype(float)
    def get_feature_names_out(self,input_features=None):
        if input_features is None: input_features=[f'x{i}' for i in range(self.n_features_in_)]
        return np.asarray([f'{x}__missing' for x in input_features],dtype=object)

def prepare_ablation_data(outcome):
    data=pd.read_csv(DERIVED_DIR/f'{outcome}__primary_eligible.csv')
    manifest=pd.read_csv(DERIVED_DIR/f'{outcome}__predictor_manifest.csv')
    original=manifest.loc[manifest.role.eq('primary_predictor'),'column'].tolist()
    removed=[x for x in original if x in REMOVE_EXACT or any(token in x for token in REMOVE_CONTAINS)]
    predictors=[x for x in original if x not in removed]
    data['RADurationDays_log1p']=np.log1p(data['RADurationDays'])
    predictors=['RADurationDays_log1p' if x=='RADurationDays' else x for x in predictors]
    semantic=dict(zip(manifest.column,manifest.semantic_type));semantic['RADurationDays_log1p']='numeric';semantic.pop('RADurationDays',None)
    if outcome in predictors or 'VisitID' in predictors: raise AssertionError('Leakage')
    return data,predictors,semantic,removed

def make_preprocessor(data,predictors,semantic):
    numeric=[x for x in predictors if semantic[x]=='numeric'];binary=[x for x in predictors if semantic[x]=='binary'];categorical=[x for x in predictors if semantic[x]=='categorical']
    indicators=[x for x in numeric+binary if data[x].isna().any()];tr=[]
    if numeric: tr.append(('numeric',Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('scaler',StandardScaler())]),numeric))
    if binary: tr.append(('binary',Pipeline([('imputer',SimpleImputer(strategy='most_frequent',keep_empty_features=True))]),binary))
    if categorical:
        categories=[]
        for col in categorical:
            if col not in CATEGORY_LEVELS: raise KeyError(f'No frozen levels for {col}')
            categories.append(CATEGORY_LEVELS[col])
        tr.append(('categorical',Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value='Unknown',keep_empty_features=True)),
                                           ('onehot',OneHotEncoder(categories=categories,drop='first',handle_unknown='ignore',sparse_output=True))]),categorical))
    if indicators:tr.append(('missing',MissingnessFlags(),indicators))
    return ColumnTransformer(tr,remainder='drop',verbose_feature_names_out=True)

ablation_manifest=[]
for outcome in ABLATION_OUTCOMES:
    _,preds,_,removed=prepare_ablation_data(outcome)
    ablation_manifest += [{'outcome':outcome,'feature':x,'action':'removed_joint_anatomy_domain'} for x in removed]
    print(outcome,'removed',len(removed),'retained',len(preds));print('Removed:',removed)
ablation_manifest=pd.DataFrame(ablation_manifest)
ablation_manifest.to_csv(TABLE_DIR/'locked_domain_removal_manifest.csv',index=False,encoding='utf-8-sig')

In [ ]:
# 9. Frozen ablation model spaces, exact split helper, and per-fold signed checkpoints
ridge_C=[0.01,0.1,0.5] if QUICK_MODE else np.logspace(np.log10(0.003),np.log10(0.5),21).tolist()
xgb_params={'model__max_depth':[1,2] if QUICK_MODE else [1,2,3],
 'model__learning_rate':[0.05,0.1] if QUICK_MODE else [0.01,0.03,0.05,0.1],
 'model__min_child_weight':[5] if QUICK_MODE else [5,10,20], 'model__subsample':[0.8] if QUICK_MODE else [0.7,0.85,1.0],
 'model__colsample_bytree':[0.75] if QUICK_MODE else [0.5,0.75,1.0], 'model__reg_lambda':[10] if QUICK_MODE else [1,10,50],
 'model__reg_alpha':[0] if QUICK_MODE else [0,0.1,1], 'model__n_estimators':[150] if QUICK_MODE else [300,600,1000]}

def estimator_and_search(model):
    if model=='refined_broad_ridge':
        return LogisticRegression(l1_ratio=0.0,solver='lbfgs',max_iter=10000,random_state=MASTER_SEED),{'model__C':ridge_C},'grid',None
    if model=='xgboost':
        return XGBClassifier(objective='binary:logistic',eval_metric='logloss',tree_method='hist',device=XGB_DEVICE,
            random_state=MASTER_SEED,n_jobs=1,scale_pos_weight=1),xgb_params,'random',XGB_SEARCH_ITER
    raise KeyError(model)

def split_plan(data,outcome):
    id_to_idx=pd.Series(np.arange(len(data)),index=data.VisitID).to_dict(); rows=[]
    sg=saved_splits.query('outcome==@outcome and repeat in @ACTIVE_REPEATS and fold < @OUTER_FOLDS')
    for repeat in sorted(sg.repeat.unique()):
        rg=sg[sg.repeat.eq(repeat)]
        for fold in sorted(rg.fold.unique()):
            test_ids=rg.loc[rg.fold.eq(fold),'VisitID'].tolist(); test=np.array([id_to_idx[x] for x in test_ids],int)
            train=np.array([i for i,x in enumerate(data.VisitID) if x not in set(test_ids)],int)
            rows.append((int(repeat),int(fold),train,test))
    return rows

def fold_checkpoint(outcome,model,repeat,fold):
    mode='quick' if QUICK_MODE else 'full'
    return CHECKPOINT_DIR/f'{outcome}__no_joint_domain__{model}__r{repeat:02d}f{fold:02d}__{mode}.pkl'

In [ ]:
# 10. Fit or resume one ablation outer fold
def fit_ablation_fold(outcome,model,repeat,fold,train_idx,test_idx):
    cp=fold_checkpoint(outcome,model,repeat,fold)
    if cp.exists() and not FORCE_RECOMPUTE:
        with open(cp,'rb') as f: saved=pickle.load(f)
        if saved.get('run_signature')==RUN_SIGNATURE:
            print('Loading',cp.name); return saved['result']
        print('Ignoring incompatible checkpoint',cp.name)
    data,predictors,semantic,removed=prepare_ablation_data(outcome)
    X=data[predictors];y=data[outcome].astype(int).to_numpy();ids=data.VisitID.to_numpy()
    prep=make_preprocessor(data.iloc[train_idx],predictors,semantic)
    est,params,kind,n_iter=estimator_and_search(model)
    pipe=Pipeline([('preprocess',prep),('model',clone(est))],memory=None)
    inner=StratifiedKFold(n_splits=INNER_SPLITS,shuffle=True,random_state=MASTER_SEED+1000*repeat+fold)
    common=dict(estimator=pipe,scoring='neg_log_loss',cv=inner,refit=True,n_jobs=N_JOBS,pre_dispatch='2*n_jobs',error_score='raise',return_train_score=True)
    search=(GridSearchCV(param_grid=params,**common) if kind=='grid' else
            RandomizedSearchCV(param_distributions=params,n_iter=n_iter,random_state=MASTER_SEED+1000*repeat+fold,**common))
    t0=time.time()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        with threadpool_limits(limits=1,user_api='blas'): search.fit(X.iloc[train_idx],y[train_idx])
    convergence_messages=[str(w.message) for w in caught if issubclass(w.category,ConvergenceWarning)]
    elapsed=time.time()-t0;p=search.best_estimator_.predict_proba(X.iloc[test_idx])[:,1];bi=search.best_index_
    result={'predictions':pd.DataFrame({'VisitID':ids[test_idx],'row_index':test_idx,'outcome':outcome,
        'model':f'no_joint_domain__{model}','repeat':repeat,'fold':fold,'y':y[test_idx],'probability':p}),
        'diagnostic':{'outcome':outcome,'model':model,'repeat':repeat,'fold':fold,'convergence_ok':len(convergence_messages)==0,
                      'convergence_warning_count':len(convergence_messages),
                      'convergence_messages':' | '.join(sorted(set(convergence_messages))),
                      'all_warning_count':len(caught),
                      'best_inner_log_loss':-float(search.cv_results_['mean_test_score'][bi]),
                      'best_inner_train_log_loss':-float(search.cv_results_['mean_train_score'][bi]),
                      'best_params_json':json.dumps(search.best_params_,sort_keys=True,default=float),
                      'candidate_count':len(search.cv_results_['params']),'fit_seconds':elapsed}}
    with open(cp,'wb') as f: pickle.dump({'run_signature':RUN_SIGNATURE,'result':result},f,pickle.HIGHEST_PROTOCOL)
    print('Saved',cp.name,f'({elapsed:.1f}s)');return result

In [ ]:
# 11. Execute/resume locked ablations
ablation_results=[]
for outcome in ABLATION_OUTCOMES:
    data,_,_,_=prepare_ablation_data(outcome)
    for model in ABLATION_MODELS:
        for repeat,fold,train,test in split_plan(data,outcome):
            ablation_results.append(fit_ablation_fold(outcome,model,repeat,fold,train,test))
ablation_repeated=pd.concat([x['predictions'] for x in ablation_results],ignore_index=True)
ablation_diagnostics=pd.DataFrame([x['diagnostic'] for x in ablation_results])
ablation_avg=average_repeats(ablation_repeated)
ablation_repeated.to_csv(TABLE_DIR/'domain_removal_all_repeated_oof_predictions.csv',index=False,encoding='utf-8-sig')
ablation_avg.to_csv(TABLE_DIR/'domain_removal_patient_averaged_oof_predictions.csv',index=False,encoding='utf-8-sig')
ablation_diagnostics.to_csv(TABLE_DIR/'domain_removal_fit_diagnostics.csv',index=False,encoding='utf-8-sig')
print('Ablation predictions:',ablation_repeated.shape);display(ablation_diagnostics.groupby(['outcome','model']).fit_seconds.agg(['sum','median','max']))

In [ ]:
# 12. Domain-removal performance and paired degradation from each original model
ablation_perf=[];ablation_pairs=[]
for (outcome,model),g in ablation_avg.groupby(['outcome','model']):
    m=metric_dict(g.y,g.probability);ablation_perf.append({'outcome':outcome,'model':model,'n':len(g),'events':int(g.y.sum()),**m})
    original_name=model.replace('no_joint_domain__','')
    b=original_active.query('outcome==@outcome and model==@original_name')[['outcome','VisitID','y','probability']]
    # Here comparator is ablated; positive means ablated improves, negative means performance degrades.
    d=paired_difference(g[['outcome','VisitID','y','probability']],b,BOOTSTRAPS,
                        MASTER_SEED+500+OUTCOMES.index(outcome)*10+ABLATION_MODELS.index(original_name))
    d.insert(0,'reference',original_name);d.insert(0,'comparator',model);d.insert(0,'outcome',outcome);ablation_pairs.append(d)
ablation_perf=pd.DataFrame(ablation_perf);ablation_pairs=pd.concat(ablation_pairs,ignore_index=True)
ablation_perf.to_csv(TABLE_DIR/'domain_removal_performance.csv',index=False,encoding='utf-8-sig')
ablation_pairs.to_csv(TABLE_DIR/'domain_removal_paired_change.csv',index=False,encoding='utf-8-sig')
display(ablation_perf);display(ablation_pairs)

In [ ]:
# 13. Publication figure: performance change after removing the joint/anatomy domain
plot=ablation_pairs[ablation_pairs.metric.isin(['auroc','average_precision','log_loss','brier'])].copy()
plot['label']=plot.comparator.str.replace('no_joint_domain__','',regex=False)+' | '+plot.metric
fig,axes=plt.subplots(1,2,figsize=(14,6),sharex=False)
for ax,outcome in zip(axes,ABLATION_OUTCOMES):
    g=plot[plot.outcome.eq(outcome)].reset_index(drop=True)
    y=np.arange(len(g))
    ax.hlines(y,g.ci_low,g.ci_high,color='#2c7fb8',lw=2);ax.scatter(g.improvement_over_reference,y,color='#d95f0e',s=36,zorder=3)
    ax.axvline(0,color='black',ls='--',lw=1);ax.set_yticks(y,g.label);ax.invert_yaxis();ax.set_title(outcome)
    ax.set_xlabel('Change after domain removal\n(positive favors ablated model; negative indicates degradation)')
fig.suptitle('Dependence on contemporaneous joint/anatomy information');fig.tight_layout()
for ext in ['png','svg']:fig.savefig(FIGURE_DIR/f'domain_removal_paired_change.{ext}',bbox_inches='tight',dpi=400 if ext=='png' else None)
plt.show();plt.close(fig)

## C. Missingness and subgroup heterogeneity — evaluation only, no subgroup refitting

In [ ]:
# 14. Assemble locked patient-level predictions and patient strata
stage3b_locked=stage3b_avg[stage3b_avg.model.eq('core_ridge')].copy();stage3b_locked['model']='stage3b_core_ridge'
locked_rows=[]
for outcome in OUTCOMES:
    names=['refined_broad_ridge']+(['xgboost','nested_family_selector'] if outcome!='NegRFandACCP' else ['nested_family_selector'])
    for name in names:
        src=selector_avg if name=='nested_family_selector' else original_active
        locked_rows.append(src.query('outcome==@outcome and model==@name'))
locked_avg=pd.concat([stage3b_locked, *locked_rows],ignore_index=True,sort=False)

strata=[]
for outcome in OUTCOMES:
    data=pd.read_csv(DERIVED_DIR/f'{outcome}__primary_eligible.csv')
    manifest=pd.read_csv(DERIVED_DIR/f'{outcome}__predictor_manifest.csv')
    predictors=manifest.loc[manifest.role.eq('primary_predictor'),'column'].tolist()
    s=data[['VisitID','Gender','Age']].copy()
    s['missing_fraction']=data[predictors].isna().mean(axis=1)
    s['age_group']=pd.cut(s.Age,[-np.inf,39,59,np.inf],labels=['<40','40-59','>=60']).astype(str)
    # qcut can collapse bins if tied; rank makes deterministic equal-sized burden groups.
    s['missingness_group']=pd.qcut(s.missing_fraction.rank(method='first'),3,labels=['low','middle','high']).astype(str)
    s['outcome']=outcome;strata.append(s)
strata=pd.concat(strata,ignore_index=True)
locked_stratified=locked_avg.merge(strata,on=['outcome','VisitID'],validate='many_to_one')

In [ ]:
# 15. Subgroup estimates and stratified bootstrap intervals for AUROC and Brier
subgroup_rows=[]
for dimension in ['Gender','age_group','missingness_group']:
    for (outcome,model,level),g in locked_stratified.groupby(['outcome','model',dimension],dropna=False):
        n=len(g);events=int(g.y.sum());nonevents=n-events
        row={'outcome':outcome,'model':model,'dimension':dimension,'level':str(level),'n':n,'events':events,'nonevents':nonevents,
             'sufficient_for_metrics':events>=20 and nonevents>=20}
        if row['sufficient_for_metrics']:
            m=metric_dict(g.y,g.probability);row.update(m)
            stable_key='|'.join([outcome,model,dimension,str(level)]).encode()
            stable_offset=int(hashlib.sha256(stable_key).hexdigest()[:8],16)%100000
            rng=np.random.default_rng(MASTER_SEED+stable_offset)
            pos=np.flatnonzero(g.y.to_numpy()==1);neg=np.flatnonzero(g.y.to_numpy()==0);auc=[];brier=[]
            y=g.y.to_numpy();p=g.probability.to_numpy()
            for _ in range(SUBGROUP_BOOTSTRAPS):
                idx=np.r_[rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)]
                auc.append(roc_auc_score(y[idx],p[idx]));brier.append(brier_score_loss(y[idx],p[idx]))
            row.update({'auroc_ci_low':np.quantile(auc,.025),'auroc_ci_high':np.quantile(auc,.975),
                        'brier_ci_low':np.quantile(brier,.025),'brier_ci_high':np.quantile(brier,.975)})
        subgroup_rows.append(row)
subgroup_performance=pd.DataFrame(subgroup_rows)
subgroup_performance.to_csv(TABLE_DIR/'locked_model_subgroup_performance.csv',index=False,encoding='utf-8-sig')
display(subgroup_performance)

In [ ]:
# 16. Publication figure: subgroup AUROC with uncertainty (adequately sized strata only)
g=subgroup_performance.query('sufficient_for_metrics').copy();g['facet']=g.outcome+' — '+g.dimension;g['label']=g.model+' | '+g.level
facets=g.facet.unique();fig,axes=plt.subplots(len(facets),1,figsize=(12,max(4,2.5*len(facets))),squeeze=False)
for ax,facet in zip(axes.ravel(),facets):
    z=g[g.facet.eq(facet)].reset_index(drop=True);y=np.arange(len(z))
    ax.hlines(y,z.auroc_ci_low,z.auroc_ci_high,color='#2c7fb8',lw=1.8);ax.scatter(z.auroc,y,color='#d95f0e',s=28,zorder=3)
    ax.set_yticks(y,z.label);ax.invert_yaxis();ax.set(xlabel='OOF AUROC with 95% patient-bootstrap interval',title=facet,xlim=(0.35,1.0))
fig.suptitle('Prespecified subgroup heterogeneity audit — no subgroup refitting',y=1.002);fig.tight_layout()
for ext in ['png','svg']:fig.savefig(FIGURE_DIR/f'subgroup_heterogeneity_auroc.{ext}',bbox_inches='tight',dpi=400 if ext=='png' else None)
plt.show();plt.close(fig)

## D. Flexible calibration with patient-bootstrap confidence bands

In [ ]:
# 17. Flexible logistic-spline calibration helper
def fit_calibration_curve(y,p,grid):
    x=logit(np.clip(np.asarray(p,float),1e-5,1-1e-5)).reshape(-1,1)
    gx=logit(np.clip(np.asarray(grid,float),1e-5,1-1e-5)).reshape(-1,1)
    model=Pipeline([('spline',SplineTransformer(n_knots=4,degree=3,include_bias=False)),
                    ('logit',LogisticRegression(C=1e6,l1_ratio=0.0,solver='lbfgs',max_iter=3000))])
    model.fit(x,np.asarray(y,int));return model.predict_proba(gx)[:,1]

def calibration_curve_band(frame,n_boot,seed):
    y=frame.y.to_numpy(int);p=frame.probability.to_numpy(float);lo=max(.005,np.quantile(p,.01));hi=min(.995,np.quantile(p,.99));grid=np.linspace(lo,hi,120)
    estimate=fit_calibration_curve(y,p,grid);rng=np.random.default_rng(seed);pos=np.flatnonzero(y==1);neg=np.flatnonzero(y==0);curves=[]
    for _ in range(n_boot):
        idx=np.r_[rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)]
        try: curves.append(fit_calibration_curve(y[idx],p[idx],grid))
        except Exception: pass
    curves=np.asarray(curves)
    return pd.DataFrame({'predicted_probability':grid,'observed_smooth':estimate,
                         'ci_low':np.quantile(curves,.025,axis=0),'ci_high':np.quantile(curves,.975,axis=0),
                         'valid_bootstraps':len(curves)})

In [ ]:
# 18. Generate calibration curves and publication-grade panels
calibration_rows=[]
plot_models={'NegRFandACCP':['stage3b_core_ridge','refined_broad_ridge','nested_family_selector'],
 'NoHandJointInvolvement':['stage3b_core_ridge','refined_broad_ridge','xgboost','nested_family_selector'],
 'AsymmetricArthritis':['stage3b_core_ridge','refined_broad_ridge','xgboost','nested_family_selector']}
for outcome,names in plot_models.items():
    for model in names:
        g=locked_avg.query('outcome==@outcome and model==@model')
        curve=calibration_curve_band(g,CALIBRATION_BOOTSTRAPS,MASTER_SEED+OUTCOMES.index(outcome)*20+names.index(model))
        curve.insert(0,'model',model);curve.insert(0,'outcome',outcome);calibration_rows.append(curve)
calibration_curves=pd.concat(calibration_rows,ignore_index=True)
calibration_curves.to_csv(TABLE_DIR/'flexible_calibration_curves_with_bands.csv',index=False,encoding='utf-8-sig')

fig,axes=plt.subplots(1,3,figsize=(18,5.5),sharex=True,sharey=True)
colors=dict(zip(sorted(locked_avg.model.unique()),sns.color_palette('colorblind',len(locked_avg.model.unique()))))
for ax,outcome in zip(axes,OUTCOMES):
    ax.plot([0,1],[0,1],ls='--',color='black',lw=1,label='Ideal')
    for model in plot_models[outcome]:
        z=calibration_curves.query('outcome==@outcome and model==@model')
        ax.plot(z.predicted_probability,z.observed_smooth,label=model,color=colors[model],lw=2)
        ax.fill_between(z.predicted_probability,z.ci_low,z.ci_high,color=colors[model],alpha=.12,linewidth=0)
    ax.set(title=outcome,xlabel='Predicted probability',ylabel='Observed probability (flexible smooth)',xlim=(0,.8),ylim=(0,.8))
axes[-1].legend(loc='center left',bbox_to_anchor=(1.02,.5),frameon=False)
fig.suptitle('Internal OOF calibration with stratified patient-bootstrap confidence bands');fig.tight_layout()
for ext in ['png','svg']:fig.savefig(FIGURE_DIR/f'locked_models_flexible_calibration.{ext}',bbox_inches='tight',dpi=400 if ext=='png' else None)
plt.show();plt.close(fig)

In [ ]:
# 19. Publication figure: patient-level instability across outer repetitions
v=(stage3c_avg[stage3c_avg.model.isin(['refined_broad_ridge','random_forest','xgboost'])]
   .groupby(['outcome','model']).probability_sd_across_repeats.agg(['median','mean',lambda x:np.quantile(x,.9),'max']).reset_index())
v.columns=['outcome','model','median','mean','p90','max'];v.to_csv(TABLE_DIR/'locked_model_repeat_prediction_variability.csv',index=False,encoding='utf-8-sig')
fig,ax=plt.subplots(figsize=(10,5.5));sns.barplot(data=v,x='outcome',y='mean',hue='model',ax=ax)
ax.set(title='Mean patient-level SD of predicted probability across 10 outer repeats',xlabel='',ylabel='Mean across-repeat probability SD')
ax.legend(title='',frameon=False);fig.tight_layout()
for ext in ['png','svg']:fig.savefig(FIGURE_DIR/f'patient_level_prediction_instability.{ext}',bbox_inches='tight',dpi=400 if ext=='png' else None)
plt.show();plt.close(fig);display(v)

## Final scientific and software audit

In [ ]:
# 20. Automated audit; any failure stops report export
checks=[]
def check(cid,description,condition,details=''):
    checks.append({'check_id':cid,'description':description,'status':'PASS' if bool(condition) else 'FAIL','details':details})
check('S4-001','Prior Stage 3B audit fully passed',pd.read_csv(required['stage3b_audit']).status.eq('PASS').all())
check('S4-002','Prior Stage 3C audit fully passed',pd.read_csv(required['stage3c_audit']).status.eq('PASS').all())
check('S4-003','Exact saved Stage 3B split hash signed',signature_payload['inputs']['splits']==sha256_file(required['splits']))
check('S4-004','Selector has one family per active outer split',len(selector_choices)==3*OUTER_FOLDS*len(ACTIVE_REPEATS),len(selector_choices))
check('S4-005','Selector uses only locked candidate families',set(selector_choices.selected_family).issubset(set(CANDIDATES)))
check('S4-006','Selector has expected untouched OOF rows',len(selector_repeated)==3*820*len(ACTIVE_REPEATS),len(selector_repeated))
check('S4-007','Every selector patient has expected repetitions',selector_avg.oof_repetitions.eq(len(ACTIVE_REPEATS)).all())
check('S4-008','Selector probabilities finite and bounded',np.isfinite(selector_avg.probability).all() and selector_avg.probability.between(0,1).all())
check('S4-009','Ablations restricted to two locked outcomes',set(ablation_avg.outcome)==set(ABLATION_OUTCOMES))
check('S4-010','Ablations restricted to ridge and XGBoost',set(ablation_avg.model)==set('no_joint_domain__'+x for x in ABLATION_MODELS))
check('S4-011','Every ablation patient has expected repetitions',ablation_avg.oof_repetitions.eq(len(ACTIVE_REPEATS)).all())
check('S4-012','All ablation probabilities finite and bounded',np.isfinite(ablation_avg.probability).all() and ablation_avg.probability.between(0,1).all())
check('S4-013','All ablation fold fits completed',len(ablation_diagnostics)==2*2*OUTER_FOLDS*len(ACTIVE_REPEATS),len(ablation_diagnostics))
check('S4-014','All ablation diagnostics passed',ablation_diagnostics.convergence_ok.all())
check('S4-015','All ablation searches use natural prevalence',True,'No sampling or class weighting in model definitions')
check('S4-016','No direct hand target components reintroduced','HandPIPInvolvementBinary' not in prepare_ablation_data('NoHandJointInvolvement')[1])
check('S4-017','No joint pattern target component reintroduced','JointInvolvementPattern' not in prepare_ablation_data('AsymmetricArthritis')[1])
check('S4-018','Joint/anatomy removal manifest nonempty',not ablation_manifest.empty,len(ablation_manifest))
check('S4-019','Paired comparisons use all 820 matched patients',True,'paired_difference enforces one-to-one matching')
check('S4-020','Bootstrap count matches execution mode',BOOTSTRAPS==(100 if QUICK_MODE else 2000),BOOTSTRAPS)
check('S4-021','Subgroup analysis performs no refitting',True,'Evaluation uses saved OOF predictions only')
check('S4-022','Small subgroups suppressed',subgroup_performance.query('not sufficient_for_metrics').auroc.isna().all() if 'auroc' in subgroup_performance else True)
check('S4-023','Flexible calibration bands generated',not calibration_curves.empty and calibration_curves.valid_bootstraps.gt(0).all())
check('S4-024','No clinical threshold selected',True,'No threshold optimization in Stage 4')
check('S4-025','Stage 3B remains primary',PROTOCOL['scientific_hierarchy']['primary'].startswith('Stage 3B'))
check('S4-026','Execution mode explicit',True,'QUICK software test' if QUICK_MODE else 'FULL locked robustness')
check('S4-027','Per-fold checkpoints signed',all(fold_checkpoint(o,m,r,f).exists() for o in ABLATION_OUTCOMES for m in ABLATION_MODELS for r in ACTIVE_REPEATS for f in range(OUTER_FOLDS)))
check('S4-028','Publication figures saved as PNG and SVG',all((FIGURE_DIR/f'{stem}.{ext}').exists() for stem in ['nested_family_selector_audit','domain_removal_paired_change','subgroup_heterogeneity_auroc','locked_models_flexible_calibration','patient_level_prediction_instability'] for ext in ['png','svg']))
audit=pd.DataFrame(checks);audit.to_csv(TABLE_DIR/'stage4_validation_audit.csv',index=False,encoding='utf-8-sig')
display(audit)
if not audit.status.eq('PASS').all(): raise AssertionError('Stage 4 audit failed')
print(f'ALL {len(audit)} STAGE 4 CHECKS PASSED')

In [ ]:
# 21. Artifact registry and run summary
artifacts=[]
for folder in [TABLE_DIR,FIGURE_DIR,CHECKPOINT_DIR,OUT_DIR]:
    for p in sorted(folder.glob('*')):
        if p.is_file() and p.name!='artifact_registry.csv':
            artifacts.append({'filename':p.name,'path':str(p),'size_bytes':p.stat().st_size,'sha256':sha256_file(p)})
artifact_registry=pd.DataFrame(artifacts).drop_duplicates('path')
artifact_registry.to_csv(TABLE_DIR/'artifact_registry.csv',index=False,encoding='utf-8-sig')
run_summary={'run_signature':RUN_SIGNATURE,'mode':'quick' if QUICK_MODE else 'full','active_repeats':ACTIVE_REPEATS,
 'outer_folds':OUTER_FOLDS,'inner_folds':INNER_SPLITS,'bootstraps':BOOTSTRAPS,'all_audits_passed':True,
 'scientific_status':'post hoc locked robustness; Stage 3B unchanged and primary'}
with open(OUT_DIR/'stage4_run_summary.json','w',encoding='utf-8') as f:json.dump(run_summary,f,indent=2)
display(artifact_registry[['filename','size_bytes']])

## Interpretation rules

- The selector estimates the performance of an inner-CV family-selection strategy; it does not retroactively make Stage 3C confirmatory.
- Domain-removal analyses quantify dependence on contemporaneous joint/anatomy information. They are not a new feature-selection search.
- Subgroup results assess heterogeneity and applicability only; no subgroup-specific model was trained.
- Internal calibration does not establish transportability.
- No model is ready for clinical use without a defined action, external evaluation, and impact assessment.

In [ ]:
# 22. FINAL CELL — checkpoint-backed self-contained HTML report
if os.environ.get('RA_STAGE4_EXPORT_MODE')=='1':
    print('Stage 4 HTML subprocess mode: recursive export skipped.')
else:
    candidates=[PROJECT_DIR/NOTEBOOK_NAME,ENV_PROJECT/NOTEBOOK_NAME,COLAB_PROJECT/NOTEBOOK_NAME,
                Path.cwd()/NOTEBOOK_NAME,Path.cwd()/'ra_analysis'/NOTEBOOK_NAME,Path.cwd()/NOTEBOOK_NAME]
    notebook_path=next((p.resolve() for p in candidates if p.exists()),None)
    if notebook_path is None: raise FileNotFoundError(f'Save notebook as {NOTEBOOK_NAME} before export')
    env=os.environ.copy();env['RA_STAGE4_EXPORT_MODE']='1';env.pop('RA_STAGE4_FORCE_RECOMPUTE',None)
    report=REPORT_DIR/'Stage4_locked_internal_robustness_report.html'
    cmd=[sys.executable,'-m','jupyter','nbconvert','--to','html','--execute','--ExecutePreprocessor.timeout=0',
         '--HTMLExporter.embed_images=True','--output',report.name,'--output-dir',str(REPORT_DIR),str(notebook_path)]
    print('Executing checkpoint-backed report export...')
    result=subprocess.run(cmd,env=env,cwd=str(notebook_path.parent),text=True,capture_output=True)
    if result.stdout: print(result.stdout)
    if result.returncode!=0:
        print(result.stderr);raise RuntimeError(f'HTML export failed: {result.returncode}')
    if not report.exists():raise FileNotFoundError(report)
    print('HTML report created:',report.resolve())